# Test bazujący na korelacji odległości

### Prezentacja 1

**Autorzy:** Igor Ciesielski, Kamil Burnecki, Michał Więcek


## Wstęp historyczny

Test korelacji odległości został wprowadzony przez Gábora J. Székely’ego w 2007 roku, jako narzędzie do wykrywania zależności między zmiennymi.

<figure>
  <img src="GaborJSzekely.jpg" alt="Gábor J. Székely">
  <figcaption>Gábor J. Székely</figcaption>
</figure>



# Do czego służy test?

Test korelacji odległości służy do sprawdzania, czy dwa wektory losowe są niezależne. <br>
W odróżnieniu od klasycznej korelacji Pearsona, metoda ta wykrywa zarówno zależności liniowe, jak i nieliniowe oraz może być stosowana do danych wielowymiarowych.

### Właściwości

1. Wykrywa zależności liniowe i nieliniowe
2. Działa dla danych jedno i wielowymiarowych

<div style="display:flex; flex-direction:column; align-items:center; gap:20px; overflow-y:auto; max-height:80vh;">

  <figure style="text-align:center;">
    <img src="Distance_Correlation_Examples.png"
         alt="Przykładowe wartości"
         style="max-width:90%; height:auto;">
    <figcaption>
      Korelacja odległościowa – wykrywa wszystkie zależności
    </figcaption>
  </figure>

  <figure style="text-align:center;">
    <img src="Pearson_Correlation_Examples.png"
         alt="Przykładowe wartości"
         style="max-width:90%; height:auto;">
    <figcaption>
      Korelacja Pearsona – wykrywa zależności liniowe
    </figcaption>
  </figure>

</div>

## Co to korelacja odległości? <sup>[1](https://arxiv.org/pdf/0803.4101)</sup>

Korelacja odległości to miara zależności między wektorami losowymi $X$ i $Y$, która wykrywa zarówno zależności liniowe, jak i nieliniowe.

### Właściwości
Zmienne $X$ i $Y$ mogą mieć dowolne wymiary $p$ i $q$ $\in N$


- Wartości korelacji odległościowej spełniają:
$$
0 \leq \operatorname{dCor}(X,Y) \leq 1
$$
W przeciwieństwie do korelacji Pearsona, korelacja odległościowa nie przyjmuje wartości ujemnych.

- Współczynnik korelacji odległościowej wynosi zero wtedy i tylko wtedy, gdy zmienne są niezależne.
$$
\operatorname{dCor}(X,Y)=0 \Longleftrightarrow X \perp Y
$$

Dla wektorów losowych $X,Y$ o funkcjach charakterystycznych $\varphi_X(t),\varphi_Y(t)$ kowariancję odległościową definiujemy jako:

$$
dCov^2(X,Y)=\int_{\mathbb{R}^{d_x+d_y}}
\frac
{|\varphi_{X,Y}(t,s)-\varphi_X(t)\varphi_Y(s)|^2}
{C_{d_x}C_{d_y}\|t\|^{1+d_x}\|s\|^{1+d_y}}
\, dt\, ds
$$

gdzie:

- $d_x,d_y$ — wymiary wektorów $X,Y$
- $C_{d_x}=\frac{\pi^{\frac{1+d_x}{2}}}{\Gamma\left(\frac{1+d_x}{2}\right)}$, -stała  
- $C_{d_y}=\frac{\pi^{\frac{1+d_y}{2}}}{\Gamma\left(\frac{1+d_y}{2}\right)}$ - stała

oraz:
- $\varphi_X(t),\varphi_Y(s)$ — funkcje charakterystyczne
- $\varphi_{X,Y}(t,s)$ — wspólna funkcja charakterystyczna zmiennych $X,Y$

### Korelacja odległościowa

$$
dCor(X,Y)=\frac{dCov(X,Y)}{\sqrt{dCov(X,X)\cdot dCov(Y,Y)}}
$$

Rozwiązanie Qingyang Zhang'a <sup>[2](https://www.sciencedirect.com/science/article/abs/pii/S0167715218304048)</sup>


$$
dCor=
\frac{
\sqrt{\sum_{i=1}^{dx}\sum_{j=1}^{dy} (\pi_{ij}-\pi_{i++}\pi_{+j})^2}
}{
\left[\sum_{i=1}^{dx} \pi_{i+}^2\left(\sum_{i=1}^{dx} \pi_{i+}^2+1\right)-2\sum_{i=1}^{dx} \pi_{i+}^3\right]^\frac{1}{4}
\cdot
\left[\sum_{j=1}^{dy} \pi_{+j}^2\left(\sum_{j=1}^{dy} \pi_{+j}^2+1\right)-2\sum_{j=1}^{dy} \pi_{+j}^3\right]^\frac{1}{4}
}
$$


In [4]:
function dCor_Zhang(x, y)
    nij = tabela(x, y)
    n = sum(nij)
    pij = nij ./ n
    pi = sum(pij, dims=2)
    pj = sum(pij, dims=1)
    pipj = (pi[i] * pj[j] for i in 1:length(pi), j in 1:length(pj))
    a = sum((e - o)^2 for (e, o) in zip(pij, pipj))
    b = sum(i -> i^2, pi) * (sum(i -> i^2, pi) + 1) - 2sum(i -> i^3, pi)
    c = sum(i -> i^2, pj) * (sum(i -> i^2, pj) + 1) - 2sum(i -> i^3, pj)
    return sqrt(a / sqrt(b * c))
end

dCor_Zhang (generic function with 1 method)

$$
T_{dCor}=
\frac{
\sqrt{\sum_{i=1}^{dx}\sum_{j=1}^{dy} (p_{ij}-p_{i++}p_{+j})^2}
}{
\left[\sum_{i=1}^{dx} p_{i+}^2\left(\sum_{i=1}^{dx} p_{i+}^2+1\right)-2\sum_{i=1}^{dx} p_{i+}^3\right]^\frac{1}{4}
\cdot
\left[\sum_{j=1}^{dy} p_{+j}^2\left(\sum_{j=1}^{dy} p_{+j}^2+1\right)-2\sum_{j=1}^{dy} p_{+j}^3\right]^\frac{1}{4}
}
$$

Rozwiązanie Székely'ego

Wymaga ono macierzy podanej wzorem:

$$
A_{kl} = a_{kl} - \bar{a}_{k·} - \bar{a}_{·l} + \bar{a}_{··}
$$

$a_{kl} = \|X_k - X_l\|_p$-odległość obserwacji

$\bar{a}_{··} = \frac{1}{n^2} \sum_{k=1}^{n} \sum_{l=1}^{n} a_{kl}$-Średnia odległość globalna

$\bar{a}_{k·} = \frac{1}{n} \sum_{l=1}^{n} a_{kl}$-Średnia odległość w wierszu

$\bar{a}_{·l} = \frac{1}{n} \sum_{k=1}^{n} a_{kl}$-Średnia odległość w kolumnie

Analogicznie definiujemy macierz $B_{kl}$ dla zmiennej Y

Wtedy:

$$
dCov^{2}_{n}(X,Y)= \frac{1}{n^2} \sum_{k=1}^{n} \sum_{l=1}^{n} A_{kl} B_{kl}
$$

A korelacja jest równa:

$$
dCor^{2}(X,Y)=\frac{dCov^{2}(X,Y)}{\sqrt{dCov^{2}(X,X)\cdot dCov^{2}(Y,Y)}}
$$

In [5]:
function dCov_Székely(x, y)
    n = length(x)
    A = zeros(Float64, n, n)
    B = zeros(Float64, n, n)
    for i in 1:n, j in 1:n
        A[i, j] = abs(x[i] - x[j])
        B[i, j] = abs(y[i] - y[j])
    end
    A = A .- mean(A, dims=1) .- mean(A, dims=2) .+ mean(A)
    B = B .- mean(B, dims=1) .- mean(B, dims=2) .+ mean(B)
    return mean(A .* B)
end

function dCor_Székely(x, y)
    return dCov_Székely(x, y) / sqrt(dCov_Székely(x, x) * dCov_Székely(y, y))
end


dCor_Székely (generic function with 1 method)

In [6]:
function dCor7(x, y)
    sum_x = sum(x)
    sum_y = sum(y)
    n = sum_x + sum_y
    nj₁ = sum_x / n
    nj₂ = sum_y / n
    z = (x .+ y) ./ n
    z2 = sum(i -> i^2, z)
    a = sum(i -> (i[1] / n - i[3] * nj₁)^2 + (i[2] / n - i[3] * nj₂)^2, zip(x, y, z))
    b = z2 * (z2 + 1) - 2sum(i -> i^3, z)
    c = (nj₁^2 + nj₂^2) * (nj₁^2 + nj₂^2 + 1) - 2(nj₁^3 + nj₂^3)
    return sqrt(a / sqrt(b * c))
end

dCor7 (generic function with 1 method)

# Możliwe zastosowania

- Analiza rynków finansowych
- Analiza biologiczna
- Astrofizyka<sup>[5](https://iopscience.iop.org/article/10.1088/0004-637X/781/1/39)</sup>-Porównywanie cech ciał kosmicznych
- Analiza danych przestrzennych<sup>[6](https://www.sciencedirect.com/science/article/pii/S1056819023013520)</sup>-analiza danych przestrzennych wyborów


## Analiza rynków finansowych <sup>[3](https://www.mdpi.com/2227-7390/11/18/3832)</sup>


Obliczono zależności między parami akcji za pomocą korelacji odległości.
Użyto korelacje odległości, ponieważ wykrywa liniowe oraz nieliniowe.

Za pomocą tego policzyli miarę odmienności dla każdej pary podaną wzorem:
$$
D(X,Y)= 1-\operatorname{dCor}(X,Y)
$$

I stworzyli minimalne drzewo rozpinające za pomocą algorytmu Prima, pokazując hierarchię rynku giełdowego
<figure>
  <img src="Tabela1_Przykład_Finansowy.png" alt="Gábor J. Székely">
</figure>


### Korelacja firm Dollartree i Autodesk
Dollartree-Sieć sklepów   <br>
Autodesk-Firma technologiczna np.AutoCAD

<figure>
  <img src="Zdjecie1_Przykład_Finansowy.png" alt="Gábor J. Székely">
</figure>

### Korelacje spółek w S&P500
<figure>
  <img src="Zdjecie2_Przykład_Finansowy.png" alt="Gábor J. Székely">
</figure>

## Analiza biologiczna <sup>[4](https://pmc.ncbi.nlm.nih.gov/articles/PMC8862277/#Abs1)</sup>

Zastąpienie klasycznego modelu WGCNA, który wykorzystuje korelacje Pearsona, wykorzystywanego do tworzenia macierzy podobieństw między genami, modelem DC-WGCNA który używa wartości korelacji odległości. Tworzymy na jego podstawie dendogramy, którę rozróżniają mniej grup(na rysunku 13 Pearson vs 11 dCor).

<figure>
  <img src="Zdjecie1_Przykład_Biologiczny.png" alt="Gábor J. Székely">
</figure>

### Odporność na outliery

<figure>
  <img src="Zdjecie2_Przykład_Biologiczny.png"
   alt="Gábor J. Székely"
    style="width:100%; height:auto;">
</figure>

Wykorzystane materiały:
1. [MEASURING AND TESTING DEPENDENCE BY CORRELATION OF DISTANCES](https://arxiv.org/pdf/0803.4101)-Gabor J. Szekely, Maria L. Rizzo and Nail K. Bakirov
2. [Independence test for large sparse contingency tables based on distance correlation](https://www.sciencedirect.com/science/article/abs/pii/S0167715218304048)-Qingyang Zhang
3. [Distance Correlation Market Graph: The Case of S&P500 Stocks](https://www.mdpi.com/2227-7390/11/18/3832)-Samuel Ugwu, Pierre Miasnikof, Yuri Lawryshyn
4. [Distance correlation application to gene co-expression network analysis](https://pmc.ncbi.nlm.nih.gov/articles/PMC8862277/#Abs1)-Jie Hou, Xiufen Ye, Weixing Feng, Qiaosheng Zhang, Yatong Han, Yusong Liu, Yu Li, Yufen Wei
5. [DISTANCE CORRELATION METHODS FOR DISCOVERING ASSOCIATIONS IN LARGE ASTROPHYSICAL DATABASES](https://iopscience.iop.org/article/10.1088/0004-637X/781/1/39)-Elizabeth Martínez-Gómez, Mercedes T. Richards, Donald St. P. Richards
6. [A distance correlation index of spatial dependence for compositional data](https://www.sciencedirect.com/science/article/pii/S1056819023013520)-M. Simona Andreano, Roberto Benedetti, Federica Piersimoni